# Assignment 3: The "Multimodal Sentiment Engine" Challenge

**Total Marks:** 20 | **Deadline:** 7:00 PM, 22nd March, 2026 | 
**Submission:** A zip file of the folder containing this notebook, and the csv/image files you will create.


---

## Setup

Run the cell below **once** to install all required packages and download NLTK data.

In [1]:
!pip install -r requirements.txt -q

import nltk
for pkg in ["wordnet", "averaged_perceptron_tagger_eng", "punkt_tab", "omw-1.4"]:
    nltk.download(pkg, quiet=True)
print("Setup complete!")

Setup complete!


In [2]:
import os, re, json, time, random, warnings
from collections import Counter
from itertools import combinations

from dotenv import load_dotenv
load_dotenv()

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag

warnings.filterwarnings("ignore")

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
LLM_MODEL = "google/gemini-2.0-flash-001"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Sentiment-bearing words to preserve during augmentation
PRESERVE_WORDS = {
    "amazing", "terrible", "awful", "excellent", "wonderful", "horrible",
    "great", "bad", "good", "worst", "best", "love", "hate", "boring",
    "fantastic", "brilliant", "pathetic", "outstanding", "dreadful",
    "superb", "mediocre",
}

print("Imports loaded. API key present:", bool(OPENROUTER_API_KEY))

Imports loaded. API key present: True


## Task 1: Data Consolidation & Classical Augmentation (5 Marks)

**Steps:**
1. Load all three CSVs and merge them
2. Train a baseline Logistic Regression on `gold_standard_100.csv` (TF-IDF features)
3. Filter `llm_labels_150.csv` -- keep only reviews where baseline confidence â‰¥ 0.65 AND agrees with LLM label
4. Deduplicate by review text $\rightarrow$ save `consolidated_base.csv`
5. Identify minority class, apply 2 augmentation methods (Synonym Replacement, Back Translation)
6. Quality filter augmented samples (Jaccard similarity)
7. Save `augmented_classical.csv` and `class_distribution.png`

In [3]:
gold = pd.read_csv("gold_standard_100.csv")
weak = pd.read_csv("weak_labels_200.csv")
llm  = pd.read_csv("llm_labels_150.csv")
print(f"Gold: {len(gold)}, Weak: {len(weak)}, LLM: {len(llm)}")

def train_baseline_model(train_df, text_col="review", label_col="label"):
    """Returns (vectorizer, classifier) trained on the given dataframe."""
    vec = TfidfVectorizer(max_features=5000, stop_words="english")
    X = vec.fit_transform(train_df[text_col])
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X, train_df[label_col])
    return vec, clf

vec, clf = train_baseline_model(gold)

#  1c. Filter LLM labels by confidence
llm_X = vec.transform(llm["review"])
llm_probs = clf.predict_proba(llm_X)
llm_confidence = llm_probs.max(axis=1)
llm_predictions = clf.predict(llm_X)

filtered_llm = llm[
    (llm_confidence >= 0.65) & (llm_predictions == llm["label"])
].copy()
print(f"Filtered LLM samples (conf >= 0.65 & agree): {len(filtered_llm)}")

#  1d. Merge & deduplicate
consolidated = pd.concat([gold, weak, filtered_llm], ignore_index=True)
consolidated = consolidated.drop_duplicates(subset="review").reset_index(drop=True)
consolidated.to_csv("consolidated_base.csv", index=False)
print(f"consolidated_base.csv saved: {len(consolidated)} unique reviews")

#  1e. Class distribution analysis
class_counts = consolidated["label"].value_counts()
minority_class = class_counts.idxmin()
print(f"Class distribution:\n{class_counts}")
print(f"Minority class: {minority_class}")

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(class_counts.index, class_counts.values,
              color=["#4CAF50", "#F44336", "#2196F3"][:len(class_counts)])
ax.set_title("Class Distribution of Consolidated Dataset", fontsize=14)
ax.set_xlabel("Sentiment")
ax.set_ylabel("Count")
for bar, val in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(val), ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150)
plt.close()
print("class_distribution.png saved.")

Gold: 100, Weak: 220, LLM: 150
Filtered LLM samples (conf >= 0.65 & agree): 27
consolidated_base.csv saved: 328 unique reviews
Class distribution:
label
Negative    151
Neutral     115
Positive     62
Name: count, dtype: int64
Minority class: Positive
class_distribution.png saved.


In [4]:

#  1f. Augmentation functions

def synonym_replacement(text, replace_frac=0.15):
    """Replace 15-20% of words with WordNet synonyms. Preserve sentiment-bearing words."""
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)

    # Map Penn POS tags to WordNet POS tags
    def get_wn_pos(tag):
        if tag.startswith("J"):
            return wordnet.ADJ
        elif tag.startswith("V"):
            return wordnet.VERB
        elif tag.startswith("N"):
            return wordnet.NOUN
        elif tag.startswith("R"):
            return wordnet.ADV
        return None

    # Identify replaceable indices: not preserve words, not punctuation
    replaceable = [
        i for i, (word, tag) in enumerate(tagged)
        if word.lower() not in PRESERVE_WORDS
        and word.isalpha()
        and get_wn_pos(tag) is not None
    ]

    n_replace = max(1, int(len(replaceable) * replace_frac))
    indices_to_replace = random.sample(replaceable, min(n_replace, len(replaceable)))

    new_tokens = tokens[:]
    for i in indices_to_replace:
        word, tag = tagged[i]
        wn_pos = get_wn_pos(tag)
        synsets = wordnet.synsets(word, pos=wn_pos)
        synonyms = []
        for syn in synsets:
            for lemma in syn.lemmas():
                candidate = lemma.name().replace("_", " ")
                if candidate.lower() != word.lower() and candidate.isalpha():
                    synonyms.append(candidate)
        if synonyms:
            new_tokens[i] = random.choice(synonyms)

    return " ".join(new_tokens)


def back_translate(text, src="en", mid="hi"):
    """Translate English â†’ Hindi â†’ English using deep-translator GoogleTranslator."""
    from deep_translator import GoogleTranslator
    try:
        hindi = GoogleTranslator(source=src, target=mid).translate(text)
        time.sleep(0.5)  # rate-limit sleep
        back = GoogleTranslator(source=mid, target=src).translate(hindi)
        time.sleep(0.5)
        return back if back else text
    except Exception as e:
        print(f"Back-translation failed: {e}")
        return text


def quality_filter(original, augmented):
    """Return True if augmented text passes Jaccard similarity (0.30â€“0.95)."""
    set_orig = set(original.lower().split())
    set_aug  = set(augmented.lower().split())
    union = set_orig | set_aug
    if not union:
        return False
    jaccard = len(set_orig & set_aug) / len(union)
    return 0.30 <= jaccard <= 0.95


#  1g. Apply augmentation to minority class
minority_samples = consolidated[consolidated["label"] == minority_class].copy()
print(f"Minority class '{minority_class}' has {len(minority_samples)} samples. Augmenting...")

augmented_rows = []
for _, row in minority_samples.iterrows():
    original_text = row["review"]
    label = row["label"]

    # Method 1: Synonym Replacement
    syn_aug = synonym_replacement(original_text, replace_frac=random.uniform(0.15, 0.20))
    if syn_aug and quality_filter(original_text, syn_aug):
        augmented_rows.append({"review": syn_aug, "label": label, "augmentation": "synonym_replacement"})

    # Method 2: Back Translation
    bt_aug = back_translate(original_text)
    if bt_aug and quality_filter(original_text, bt_aug):
        augmented_rows.append({"review": bt_aug, "label": label, "augmentation": "back_translation"})

augmented_df = pd.DataFrame(augmented_rows)
print(f"Augmented samples generated (after quality filter): {len(augmented_df)}")

# Combine original consolidated with augmented minority samples
augmented_classical = pd.concat(
    [consolidated.assign(augmentation="original"), augmented_df],
    ignore_index=True
)
augmented_classical = augmented_classical.drop_duplicates(subset="review").reset_index(drop=True)
augmented_classical.to_csv("augmented_classical.csv", index=False)
print(f"augmented_classical.csv saved: {len(augmented_classical)} total rows")

Minority class 'Positive' has 62 samples. Augmenting...
Augmented samples generated (after quality filter): 123
augmented_classical.csv saved: 449 total rows


## Task 2: LLM-Based Synthetic Review Generation (5 Marks)

**Steps:**
1. Design a few-shot prompt with 3-4 gold-standard examples
2. Use OpenRouter API (via `openai` package) to generate 300 synthetic reviews in batches of 20
3. Calculate diversity metrics: Self-BLEU per class
4. Run sentiment consistency check with baseline model $\rightarrow$ flag mismatches
5. Save `llm_generated_300.csv`, `llm_generated_flagged.csv`, `prompt_template.txt`, `diversity_report.txt`

In [6]:
from openai import OpenAI

client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)

#  2a. Design your few-shot prompt 
# TODO: Build a prompt string with 3-4 example reviews from gold standard
# Instruct the LLM to output JSON: [{"review": "...", "sentiment": "Positive", "movie": "..."}]
PROMPT_TEMPLATE = """You are a movie review generator. ..."""

# Save prompt to file
with open("prompt_template.txt", "w", encoding="utf-8") as f:
    f.write(PROMPT_TEMPLATE)

#  2b. Generate reviews in batches 
# TODO: Loop to generate ~300 reviews in batches of 20
# Target distribution: ~150 Positive, ~100 Negative, ~50 Neutral
# Parse JSON response, handle errors


#  2c. Diversity metrics 
# TODO: Calculate Self-BLEU per sentiment class using nltk.translate.bleu_score


#  2d. Sentiment consistency check 
# TODO: Use baseline model (vec, clf) to predict sentiment of each generated review
# TODO: Flag mismatches, save llm_generated_flagged.csv


#  2e. Save outputs 
# TODO: Save llm_generated_300.csv and diversity_report.txt

## Task 3: Multilingual Sentiment Translation (4 Marks)

**Steps:**
1. Sample 100 reviews (40 Pos, 40 Neg, 20 Neu), prioritize shorter reviews
2. Translate English $\rightarrow$ Hindi using `deep-translator` (`GoogleTranslator`)
3. Back-translate Hindi $\rightarrow$ English, compute BLEU score (threshold â‰¥ 0.3)
4. Check sentiment preservation on back-translated text
5. Manually verify 5 random samples
6. Save `bilingual_reviews.csv` with `bleu_score` and `quality_flag` columns

In [ ]:
from operator import length_hint

from _pytest._code import source
from numpy.random import random_sample
from sklearn.preprocessing import TargetEncoder

from deep_translator import GoogleTranslator
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import pandas as pd
import time
import random
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

consolidated_base = pd.read_csv("consolidated_base.csv")
consolidated_base['label'] = consolidated_base['label'].str.strip().str.capitalize()

from sklearn.feature_extraction.text import Tfisdfvectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

gold = pd.read_csv("gold_standard_100.csv")
gold['label'] = gold['label'].str.strip().str.capitalize()

baseline_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000, sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=1000, C=1.0, random_state=42))
])
baseline_model.fit(gold['review'], gold['label'])

#  3a. Strategic sampling 
# TODO: From consolidated_base, sample 100 reviews (40 Pos, 40 Neg, 20 Neu)
# Prioritize shorter reviews (sort by length, take shortest)

def sample_by_class(df, label, n):
    subset = df[df['label'] == label].copy()
    subset = subset.assign(length=subset['review'].str.len())
    subset = subset.sort_values('length')
    return subset.head(n)[['review', 'label']]

pos_samples = sample_by_class(consolidated_base, 'Positive', 40)
neg_samples = sample_by_class(consolidated_base, 'Negative', 40)
neu_samples = sample_by_class(consolidated_base, 'Neutral', 20)

sampled_100 = pd.concat([pos_samples, neg_samples, neu_samples], ignore_index=True)

print(f"sampled {len(sampled_100)} reviews -> "
      f"positive: {len(pos_samples)} negative: {len(neg_samples)} neutral: {len(neu_samples)}")


#  3b. Translation pipeline 
# TODO: Translate each review English $\rightarrow$ Hindi using GoogleTranslator(source='en', target='hi')
# Add time.sleep(0.5) between calls to avoid rate limits

hindi_translations = []

for i, review in enumerate(sampled_100['review']):
    try:
        hindi = GoogleTranslator(source='en', target='hi').translate(review)
        hindi_translations.append(hindi if hindi else review)
    except Exception as e:
        print(f"[row{i}] translation failed: {e}")
        hindi_translations.append(review)
        time.sleep(0.5)

sampled_100['hindi'] = hindi_translations


#  3c. Back-translation & BLEU 
# TODO: Translate Hindi $\rightarrow$ English
# Compute sentence BLEU between original and back-translated
# quality_flag = "PASS" if BLEU >= 0.3, else "FAIL"

smoother = SmoothingFunction().method1

back_translations = []
bleu_scores = []
quality_flags = []

for i, row in sampled_100.iterrows():
    try:
        back_en = GoogleTranslator(source='hi', target='en').translate(row['hindi'])
        back_en = back_en -f back_en else row['review']
    except Exception as e:
        print(f"[row{i}] back-translation failed:{e}")
        back_en = row['review']
    time.sleep(0.5)

    ref_tokens = nltk.word_tokenize(row['review'].lower())
    hyp_tokens = nltk.word_tokenize([back_en].lower())
    bleu = sentence_bleu([ref_tokens], hyp_tokens, weights=(0.5,0.5), smoothing_function=smoother)
    bleu = round(float(bleu), 4)

    back_translations.append(back_en)
    bleu_scores.append(bleu)
    quality_flags.append("PASS" if bleu >= 0.3 else "FAIL")

sampled_100['back_translated'] = back_translations
sampled_100['bleu_score'] = bleu_scores
sampled_100['quality_flag'] = quality_flags

prin(f"\nBLEU quality -> PASS:{quality_flags.count('PASS')}"
     f"FAIL: {quality_flags.count('FAIL')}"
     f"(avg BLEU: {sum(bleu_scores)/len(bleu_scores):.4f})")


#  3d. Sentiment preservation check 
# TODO: Predict sentiment on back-translated text, compare with original label

pred_back = baseline_model.predict(sampled_100['back_translated'].tolist())
sampled_100['predicted_sentiment'] = pred_back
sampled_100['sentiment_preserved'] = (
    sampled_100['label'].str.lower() == sampled_100['predicted_sentiment'].str.lower()
)

preserved_count = sampled_100['sentiment_preserved'].sum()
print(f"Sentiment preserved: {preserved_count}/{len(sampled_100)}"
      f"({100*preserved_count/len(sampled_100):.1f}%)")


#  3e. Manual verification 
# TODO: Print 5 random samples for inspection

random.seed(42)
five_indices = random.sample(range(len(sampled_100)), 5)

print("\n --Manual Verification(5 random samples)--")

for rank, idx in enumerate(five_indices, 1):
    row = sampled_100.iloc[idx]
    print(f"\nSample {rank}:")
    print(f" Original [{row['label']}] : {row['review']}")
    print(f" Hindi                     : {row['review']}")
    print(f" Back-EN                   : {row['back_translated']}")
    print(f" BLEU: {row['bleu_score']:.4f} | Quality:{row['quality_flag']}"
          f"| Sentiment preserved: {row['sentiment_preserved']}")


#  3f. Save 
# TODO: Save bilingual_reviews.csv with columns:
# review, label, hindi, back_translated, bleu_score, quality_flag

output_cols = ['review', 'label', 'hindi', 'back_translated', 'bleu_score', 'quality_flag']
sampled_100[output_cols].to_csv("bilingual_reviews.csv", index=False)

print(f"\nSaved bilingual_reviews.csv ({len(sampled_100)} rows)")
print(f"Columns: {output_cols}")

## Task 4: Multimodal Audio Generation (4 Marks)

**Steps:**
1. Select 30 reviews (10 per class) of varying lengths
2. Use `gTTS` to generate audio (`tld="com"`), convert mp3$\rightarrow$wav via `librosa`+`soundfile`
3. Extract audio features with `librosa`: duration, spectral centroid, zero crossing rate, MFCCs
4. Use `openai-whisper` (tiny model) to transcribe audio back to text, compute WER
5. Save `audio_samples/` folder, `audio_features.csv`, `audio_validation.csv`

In [14]:
from gtts import gTTS
import librosa, soundfile as sf

#  4a. Select 30 reviews (10 per class)
# TODO: Sample from consolidated_base, mix short and long reviews.
sampled_parts = []
per_class_target = 10

rng = np.random.default_rng()
sentiment_order = list(consolidated['label'].dropna().unique())
rng.shuffle(sentiment_order)

for sentiment in sentiment_order:
    class_samples = consolidated[consolidated['label'] == sentiment].copy()
    class_samples = class_samples.dropna(subset=['review']).drop_duplicates(subset=['review'])
    class_samples['review_len'] = class_samples['review'].str.len()

    short_pool = class_samples[class_samples['review_len'] <= 100]
    long_pool = class_samples[class_samples['review_len'] > 100]

    short_n = min(5, len(short_pool))
    long_n = min(5, len(long_pool))

    short_samples = short_pool.sample(short_n) if short_n else short_pool.head(0)
    long_samples = long_pool.sample(long_n) if long_n else long_pool.head(0)

    selected = pd.concat([short_samples, long_samples])

    if len(selected) < per_class_target:
        remaining = class_samples.drop(index=selected.index, errors='ignore')
        fill_n = min(per_class_target - len(selected), len(remaining))
        if fill_n:
            fill_samples = remaining.sample(fill_n)
            selected = pd.concat([selected, fill_samples])

    selected = selected.sample(frac=1).head(per_class_target) if len(selected) else selected
    print(f"Selected {len(selected)} samples for sentiment {sentiment}")
    sampled_parts.append(selected[['review', 'label']])

sampled = pd.concat(sampled_parts, ignore_index=True)
sampled = sampled.drop_duplicates(subset=['review']).reset_index(drop=True)

if len(sampled) > 30:
    sampled = sampled.groupby('label', group_keys=False).apply(lambda x: x.sample(min(10, len(x)))).reset_index(drop=True)

os.makedirs("audio_samples", exist_ok=True)
sampled.to_csv("audio_samples/sampled_reviews.csv", index=False)


#  4b. TTS generation
# TODO: For each review, generate audio with gTTS (tld="com").
# TODO: Save as mp3, then load with librosa and re-save as .wav via soundfile.

for directory in ["mp3", "wav"]:
    dir_path = os.path.join("audio_samples", directory)
    if os.path.exists(dir_path):
        for file in os.listdir(dir_path):
            file_path = os.path.join(dir_path, file)
            if os.path.isfile(file_path):
                os.remove(file_path)
    else:
        os.makedirs(dir_path, exist_ok=True)

for idx, row in sampled.iterrows():
    review_text = str(row['review'])
    sentiment = str(row['label'])
    mp3filename_base = f"audio_samples/mp3/review_{idx}_{sentiment}"
    wavfilename_base = f"audio_samples/wav/review_{idx}_{sentiment}"

    tts = gTTS(text=review_text, lang='en', tld='com')
    mp3_path = f"{mp3filename_base}.mp3"
    wav_path = f"{wavfilename_base}.wav"

    tts.save(mp3_path)
    audio, sr = librosa.load(mp3_path, sr=16000)
    sf.write(wav_path, audio, sr)

print("Audio generation complete:", len(sampled), "samples")

Selected 10 samples for sentiment Positive
Selected 10 samples for sentiment Negative
Selected 10 samples for sentiment Neutral
Audio generation complete: 30 samples


In [15]:

#  4c. Audio feature extraction 
# TODO: For each wav, extract with librosa:
#   - duration (librosa.get_duration)
#   - spectral centroid (librosa.feature.spectral_centroid)
#   - zero crossing rate (librosa.feature.zero_crossing_rate)
#   - MFCCs (librosa.feature.mfcc, n_mfcc=13, take mean)
# Save audio_features.csv

audio_rows = []

for wav_file in sorted(os.listdir("audio_samples/wav")):
    if not wav_file.lower().endswith(".wav"):
        continue
    
    wav_path = os.path.join("audio_samples/wav", wav_file)
    filename = os.path.basename(wav_path)

    stem = os.path.splitext(filename)[0]
    parts = stem.split("_", 2)
    sentiment = parts[2] if len(parts) == 3 else "Unknown"

    try:
        audio, sr = librosa.load(wav_path, sr=16000)
        duration = librosa.get_duration(y=audio, sr=sr)
        spectral_centroid = float(np.mean(librosa.feature.spectral_centroid(y=audio, sr=sr)))
        zero_crossing_rate = float(np.mean(librosa.feature.zero_crossing_rate(y=audio)))
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
        mfcc_mean = float(np.mean(mfccs))

        audio_rows.append({
            "filename": filename,
            "sentiment": sentiment,
            "duration": duration,
            "spectral_centroid": spectral_centroid,
            "zero_crossing_rate": zero_crossing_rate,
            "mfcc_mean": mfcc_mean,
        })
    except Exception as e:
        print(f"Skipping {filename}: {e}")

audio_data = pd.DataFrame(audio_rows, columns=[
    "filename", "sentiment", "duration", "spectral_centroid", "zero_crossing_rate", "mfcc_mean"
])
audio_data.to_csv("audio_features.csv", index=False)
print("audio_features.csv rows:", len(audio_data))
audio_data.head()

audio_features.csv rows: 30


,filename,sentiment,duration,spectral_centroid,zero_crossing_rate,mfcc_mean
0,review_0_Positive.wav,Positive,5.376,1583.250772,0.114873,-22.182865
1,review_10_Negative.wav,Negative,5.424,1630.887417,0.125304,-22.317148
2,review_11_Negative.wav,Negative,4.728,2102.452592,0.182258,-21.745945
3,review_12_Negative.wav,Negative,7.584,1989.537246,0.169485,-21.283775
4,review_13_Negative.wav,Negative,7.872,1613.939030,0.119384,-20.569633


In [16]:
#  4d. Whisper round-trip validation
# TODO: Transcribe each wav with Whisper.
# TODO: Compute WER (word-level Levenshtein distance / reference word count).
# TODO: Flag samples with WER > 0.25 and save audio_validation.csv.
import whisper

_whisper_model = whisper.load_model("tiny")

def _word_levenshtein_distance(ref_words, hyp_words):
    """Compute Levenshtein edit distance between two word lists."""
    m, n = len(ref_words), len(hyp_words)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if ref_words[i - 1] == hyp_words[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost,
            )
    return dp[m][n]

def compute_wer(reference, hypothesis):
    ref_words = re.findall(r"\w+", str(reference).lower())
    hyp_words = re.findall(r"\w+", str(hypothesis).lower())
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else 1.0
    dist = _word_levenshtein_distance(ref_words, hyp_words)
    return dist / len(ref_words)

if 'sampled' in globals() and {'review', 'label'}.issubset(sampled.columns):
    sampled_for_lookup = sampled[['review', 'label']].reset_index(drop=True)
elif os.path.exists("audio_samples/sampled_reviews.csv"):
    sampled_for_lookup = pd.read_csv("audio_samples/sampled_reviews.csv")[['review', 'label']].reset_index(drop=True)
else:
    raise FileNotFoundError(
        "Missing sampled metadata. Run Task 4 cell 12 first to generate audio_samples/sampled_reviews.csv."
    )

sampled_lookup = sampled_for_lookup.to_dict("index")
validation_rows = []

for wav_file in sorted(os.listdir("audio_samples/wav")):
    if not wav_file.lower().endswith(".wav"):
        continue

    wav_path = os.path.join("audio_samples/wav", wav_file)

    base = os.path.splitext(wav_file)[0]
    parts = base.split("_", 2)
    if len(parts) < 3 or parts[0] != "review":
        continue

    try:
        sample_idx = int(parts[1])
    except ValueError:
        continue

    if sample_idx not in sampled_lookup:
        continue

    reference_text = sampled_lookup[sample_idx]["review"]
    true_label = sampled_lookup[sample_idx]["label"]

    try:
        audio_wave, sr = librosa.load(wav_path, sr=16000)
        result = _whisper_model.transcribe(audio_wave, fp16=False)
        transcript = result.get("text", "").strip()
        wer = compute_wer(reference_text, transcript)
        quality_flag = "FAIL" if wer > 0.25 else "PASS"
    except Exception as e:
        transcript = ""
        wer = np.nan
        quality_flag = f"ERROR: {type(e).__name__}"
        print(f"Transcription failed for {wav_file}: {e}")

    validation_rows.append({
        "filename": wav_file,
        "label": true_label,
        "reference": reference_text,
        "transcript": transcript,
        "wer": round(wer, 4),
        "quality_flag": quality_flag,
    })

audio_validation = pd.DataFrame(validation_rows)
audio_validation.to_csv("audio_validation.csv", index=False)

print("Whisper validation complete.")
print("Total files processed:", len(audio_validation))
if len(audio_validation) and "wer" in audio_validation.columns:
    print("High-WER files (>0.25):", int((audio_validation["wer"] > 0.25).sum()))
audio_validation.head()

Whisper validation complete.
Total files processed: 30
High-WER files (>0.25): 0


,filename,label,reference,transcript,wer,quality_flag
0,review_0_Positive.wav,Positive,The lead actor delivered a delightful performa...,The lead actor delivered a delightful performa...,0.2000,PASS
1,review_10_Negative.wav,Negative,This might be the worst thing I've seen all ye...,This might be the worst thing I've seen earlie...,0.1250,PASS
2,review_11_Negative.wav,Negative,I struggled to sit through the first half. A c...,I struggle to sit through the first half. A co...,0.1818,PASS
3,review_12_Negative.wav,Negative,A frustrating experience. The soundtrack distr...,of frustrating experience. The soundtrack dist...,0.0769,PASS
4,review_13_Negative.wav,Negative,"The cinematography was mediocre, but nothing s...","The cinematography was mediocre, but nothing s...",0.0000,PASS


## Task 5: Final Dataset Assembly & Model Evaluation (2 Marks)

**Steps:**
1. Merge all datasets: `consolidated_base.csv` + `augmented_classical.csv` + `llm_generated_300.csv` (excluding flagged) + English text from `bilingual_reviews.csv`
2. Deduplicate $\rightarrow$ save `final_augmented_dataset.csv`
3. Use `BlackBoxEvaluator` from `evaluator.py` to compare baseline vs augmented accuracy

In [10]:
from evaluator import BlackBoxEvaluator

#  5a. Assemble final dataset 
# TODO: Load consolidated_base, augmented_classical, llm_generated_300, bilingual_reviews
# Exclude flagged LLM reviews
# Merge all, deduplicate on "review" column
# Save final_augmented_dataset.csv


#  5b. Black-Box Evaluation 
evaluator = BlackBoxEvaluator()
test_df = pd.read_csv("gold_standard_100.csv")

# Baseline evaluation (small dataset only)
# TODO: baseline_acc = evaluator.run_evaluation(consolidated_base, test_df)

# Augmented evaluation (full dataset)
# TODO: augmented_acc = evaluator.run_evaluation(final_augmented, test_df)

# Print comparison
# print(f"Baseline accuracy:  {baseline_acc:.2%}")
# print(f"Augmented accuracy: {augmented_acc:.2%}")
# print(f"Improvement:        {augmented_acc - baseline_acc:+.2%}")

Initializing Black-Box Embedder...


ValueError: The provided filename text_embedder.pt does not exist